In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression


# Load dataset

df = pd.read_csv("Dataset1.csv")

df = df.sort_values(["Train_No", "SN"])


# Convert time columns

df["Arrival_time"] = pd.to_datetime(
    df["Arrival_time"],
    format="%H:%M:%S"
)

df["Departure_Time"] = pd.to_datetime(
    df["Departure_Time"],
    format="%H:%M:%S"
)


# Prepare model dataset

model_data = df.groupby("Train_No").agg(
    Total_Distance=("Distance", "max"),
    Number_of_Stops=("Station_Name", "count"),
    Start_Time=("Departure_Time", "first"),
    End_Time=("Arrival_time", "last")
).reset_index()


# Handle next day journey

model_data.loc[
    model_data["End_Time"] < model_data["Start_Time"],
    "End_Time"
] += pd.Timedelta(days=1)


# Target variable

model_data["Journey_Hours"] = (
    model_data["End_Time"] -
    model_data["Start_Time"]
).dt.total_seconds()/3600



# Input features

X = model_data[
    [
        "Total_Distance",
        "Number_of_Stops"
    ]
]


# Output

y = model_data["Journey_Hours"]



# Split data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)



# Train Linear Regression model

model = LinearRegression()

model.fit(
    X_train,
    y_train
)



print("Linear Regression Model Trained Successfully")


print("\nModel Coefficients:")

coef = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

display(coef)


print("Intercept:", model.intercept_)

Linear Regression Model Trained Successfully

Model Coefficients:


,Feature,Coefficient
0,Total_Distance,0.004577
1,Number_of_Stops,0.110197


Intercept: 1.1427702480220079
